In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window 

In [0]:
df = spark.read.csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",header=True, inferSchema=True)

In [0]:
df.select("event_type").distinct().show()
df.select("price").summary().show()

+----------+
|event_type|
+----------+
|  purchase|
|      cart|
|      view|
+----------+

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          67501979|
|   mean|292.45931656479536|
| stddev|355.67449958606727|
|    min|               0.0|
|    25%|             69.24|
|    50%|            165.77|
|    75%|            360.34|
|    max|           2574.07|
+-------+------------------+



In [0]:
purchases = df.where((df.event_type == "purchase") &  (df.price.isNotNull()))

In [0]:
purchases = purchases.withColumn("revenue", F.round(F.col("price"), 2))


In [0]:
purchases.show(3)

+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+-------+
|         event_time|event_type|product_id|        category_id|       category_code|  brand| price|  user_id|        user_session|revenue|
+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+-------+
|2019-11-01 00:00:41|  purchase|  13200605|2053013557192163841|furniture.bedroom...|   NULL| 566.3|559368633|d6034fa2-41fb-4ac...|  566.3|
|2019-11-01 00:01:04|  purchase|   1005161|2053013555631882655|electronics.smart...| xiaomi|211.92|513351129|e6b7ce9b-1938-4e2...| 211.92|
|2019-11-01 00:04:51|  purchase|   1004856|2053013555631882655|electronics.smart...|samsung|128.42|562958505|0f039697-fedc-40f...| 128.42|
+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+-------+
only showing top 3 rows


In [0]:
# find high value product
product_perf = purchases.groupBy("product_id") \
    .agg(
        F.count("*").alias("purchase_count"),
        f.round(F.sum("revenue"),2).alias("total_revenue")
    )


In [0]:
product_perf.show(3)

+----------+--------------+-------------+
|product_id|purchase_count|total_revenue|
+----------+--------------+-------------+
|   1005159|          2600|    524295.44|
|   8500290|            40|      10602.3|
|  26404407|            13|      1268.28|
+----------+--------------+-------------+
only showing top 3 rows


In [0]:
#filter and order to find the top products 
top_products = product_perf.filter(F.col("total_revenue") > 1000).orderBy(F.col("total_revenue").desc())


In [0]:
#top 5 products by total revenue
top_products.show(5)

+----------+--------------+-------------+
|product_id|purchase_count|total_revenue|
+----------+--------------+-------------+
|   1005115|         22244|2.062557432E7|
|   1005105|          8483|1.144535469E7|
|   1005135|          4284|   7086522.13|
|   1004249|          8881|   6815294.62|
|   1002544|         11678|   5603193.59|
+----------+--------------+-------------+
only showing top 5 rows


In [0]:
#window function to rank the number of user purchase
user_window = Window.partitionBy("user_id").orderBy("event_time")

events_with_sequence = df.withColumn("event_sequence",F.row_number().over(user_window))


In [0]:
display(events_with_sequence.show(3))

+-------------------+----------+----------+-------------------+--------------------+--------+------+---------+--------------------+--------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand| price|  user_id|        user_session|event_sequence|
+-------------------+----------+----------+-------------------+--------------------+--------+------+---------+--------------------+--------------+
|2019-11-29 14:47:32|      view|   1004740|2053013555631882655|electronics.smart...|  xiaomi|239.36| 94584874|e6abb356-512a-447...|             1|
|2019-11-26 05:31:47|      view|   4900378|2053013555220840837|appliances.kitche...|scarlett|112.99|122384079|c04d12ef-da1c-4e4...|             1|
|2019-11-28 08:25:09|      view|  12702958|2053013553559896355|                NULL|cordiant| 42.47|122384079|6cee7edb-68ae-4be...|             2|
+-------------------+----------+----------+-------------------+--------------------+--------+------+---------+--------

In [0]:
#Cumulative spend per user
spend_window = Window.partitionBy("user_id").orderBy("event_time").rowsBetween(Window.unboundedPreceding, Window.currentRow)

events_with_spend = purchases.withColumn(
    "running_spend",
    F.sum("revenue").over(spend_window)
)


In [0]:
events_with_spend.show(5)

+-------------------+----------+----------+-------------------+-------------+---------+------+---------+--------------------+-------+-------------+
|         event_time|event_type|product_id|        category_id|category_code|    brand| price|  user_id|        user_session|revenue|running_spend|
+-------------------+----------+----------+-------------------+-------------+---------+------+---------+--------------------+-------+-------------+
|2019-11-22 20:29:48|  purchase|  12703494|2053013553559896355|         NULL| cordiant| 41.19|311699893|43417fd7-52b6-4c0...|  41.19|        41.19|
|2019-11-13 11:09:23|  purchase|  26403383|2053013563651392361|         NULL|     NULL|335.92|346447936|50a53046-6eb8-45f...| 335.92|       335.92|
|2019-11-01 18:26:28|  purchase|   5301287|2053013563173241677|         NULL|panasonic| 41.16|384989212|b610a90d-2b5a-4c3...|  41.16|        41.16|
|2019-11-21 07:42:42|  purchase|  10600011|2053013561554240247|         NULL|     NULL| 205.9|395378381|a52f96f0

In [0]:
#join
products = events_with_spend.select(
    "product_id",
    "category_code",
    "brand"
).dropDuplicates(["product_id"])

enriched_events = purchases.join(
    products,
    on="product_id",
    how="left"
)


In [0]:
display(enriched_events.show(3))

+----------+-------------------+----------+-------------------+--------------------+-------+------+---------+--------------------+-------+--------------------+-------+
|product_id|         event_time|event_type|        category_id|       category_code|  brand| price|  user_id|        user_session|revenue|       category_code|  brand|
+----------+-------------------+----------+-------------------+--------------------+-------+------+---------+--------------------+-------+--------------------+-------+
|   1005161|2019-11-01 00:01:04|  purchase|2053013555631882655|electronics.smart...| xiaomi|211.92|513351129|e6b7ce9b-1938-4e2...| 211.92|electronics.smart...| xiaomi|
|  13200605|2019-11-01 00:00:41|  purchase|2053013557192163841|furniture.bedroom...|   NULL| 566.3|559368633|d6034fa2-41fb-4ac...|  566.3|furniture.bedroom...|   NULL|
|  26401669|2019-11-01 00:05:34|  purchase|2053013563651392361|                NULL|lucente|109.66|541854711|c41c44d5-ef9b-41b...| 109.66|                NULL|l

In [0]:
#Count events by category
category_events = df.groupBy(
    "category_code",
    "event_type"
).agg(F.count("*").alias("event_count"))

#Separate views and purchases
views = category_events.filter(F.col("event_type") == "view") \
    .select("category_code", F.col("event_count").alias("views"))

buys = category_events.filter(F.col("event_type") == "purchase") \
    .select("category_code", F.col("event_count").alias("purchases"))


In [0]:
display(views)

display(buys)

category_code,views
furniture.living_room.sofa,417428
appliances.environment.fan,3316
auto.accessories.anti_freeze,3397
auto.accessories.radar,47145
electronics.audio.microphone,44645
electronics.clocks,1994440
appliances.kitchen.meat_grinder,242604
apparel.underwear,47145
kids.swing,57430
furniture.bathroom.toilet,28060


category_code,purchases
apparel.jumper,82
stationery.cartrige,191
apparel.sock,19
kids.swing,482
electronics.audio.music_tools.piano,696
appliances.kitchen.coffee_machine,724
furniture.living_room.chair,571
appliances.environment.air_heater,3583
null,234218
auto.accessories.player,3793


In [0]:
#calculate conversion rate 
conversion = views.join(buys, "category_code", "left") \
    .withColumn(
        "conversion_rate",
        F.round((F.col("purchases") / F.col("views")) * 100, 2)
    )
display(conversion)


category_code,views,purchases,conversion_rate
stationery.cartrige,11943,191,1.6
electronics.video.tv,2071305,30274,1.46
accessories.wallet,68909,366,0.53
appliances.kitchen.juicer,71894,733,1.02
null,20837460,null,null
construction.tools.welding,141960,1119,0.79
appliances.environment.air_heater,269283,3583,1.33
country_yard.furniture.hammok,1567,4,0.26
apparel.shoes,1836676,10140,0.55
electronics.audio.microphone,44645,489,1.1


In [0]:
# Spark UDF
def price_band(price):
    if price is None:
        return "UNKNOWN"
    elif price < 50:
        return "LOW"
    elif price < 200:
        return "MEDIUM"
    else:
        return "HIGH"
    
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

price_band_udf = udf(price_band, StringType())

events_with_price_band = df.withColumn(
    "price_band",
    price_band_udf("price")
)

events_with_price_band.select(
    "price", "price_band"
).show(10, truncate=False)


+------+----------+
|price |price_band|
+------+----------+
|489.07|HIGH      |
|293.65|HIGH      |
|28.31 |LOW       |
|712.87|HIGH      |
|183.27|MEDIUM    |
|360.09|HIGH      |
|514.56|HIGH      |
|30.86 |LOW       |
|72.72 |MEDIUM    |
|732.07|HIGH      |
+------+----------+
only showing top 10 rows
